# Подготовка данных для Set Transformer

Преобразование данных в формат для обучения множественных моделей (Set Transformer, Deep Sets)

## 1. Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = 'data/'

train = pd.read_csv(f'{DATA_PATH}/daimler_mixtures_train.csv')
test = pd.read_csv(f'{DATA_PATH}/daimler_mixtures_test.csv')
props = pd.read_csv(f'{DATA_PATH}/daimler_component_properties.csv')

print(f"Train: {train.shape}, Test: {test.shape}, Props: {props.shape}")

## 2. Определение колонок

In [ ]:
# Целевые переменные
TARGET_VISC = 'Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %'
TARGET_OXID = 'Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm'

# Параметры теста
TEMP_COL = "Температура испытания | ASTM D445 Daimler Oxidation Test (DOT), °C"
TIME_COL = "Время испытания | - Daimler Oxidation Test (DOT), ч"
BIO_COL = "Количество биотоплива | - Daimler Oxidation Test (DOT), % масс"
CAT_COL = "Дозировка катализатора, категория"

# Остальные колонки
COMP_COL = 'Компонент'
BATCH_COL = 'Наименование партии'
MASS_COL = 'Массовая доля, %'

SCENARIO_COL = 'scenario_id'
print("Колонки определены")

## 3. Создание справочника свойств компонентов

In [ ]:
# Преобразуем свойства в wide format
def create_properties_dict(props_df):
    """Создает словарь свойств для каждой пары компонент-партия"""
    props_dict = {}
    
    # Сначала собираем typical значения
    typical = props_df[props_df[BATCH_COL].astype(str).str.lower() == 'typical']
    typical_dict = {}
    for _, row in typical.iterrows():
        comp = row[COMP_COL]
        prop_name = row['Наименование показателя']
        prop_value = row['Значение показателя']
        if pd.notna(prop_value):
            if comp not in typical_dict:
                typical_dict[comp] = {}
            typical_dict[comp][prop_name] = prop_value
    
    # Собираем измеренные значения
    for _, row in props_df.iterrows():
        comp = row[COMP_COL]
        batch = row[BATCH_COL]
        prop_name = row['Наименование показателя']
        prop_value = row['Значение показателя']
        
        if pd.isna(batch) or pd.isna(prop_value):
            continue
            
        key = (comp, batch)
        if key not in props_dict:
            props_dict[key] = {}
        props_dict[key][prop_name] = prop_value
    
    return props_dict, typical_dict

props_dict, typical_dict = create_properties_dict(props)
print(f"Измеренных свойств: {len(props_dict)}, Typical: {len(typical_dict)}")

## 4. Получение свойств для компонента

In [ ]:
def get_component_properties(comp, batch, props_dict, typical_dict):
    """Получает свойства для компонента: сначала измеренные, потом typical"""
    # Пробуем найти точное совпадение
    key = (comp, batch)
    if key in props_dict:
        return props_dict[key]
    
    # Пробуем найти по компоненту без партии
    for k, v in props_dict.items():
        if k[0] == comp:
            return v
    
    # Используем typical
    if comp in typical_dict:
        return typical_dict[comp]
    
    return {}

# Тест
test_comp = 'Детергент_1'
test_batch = '0174.22'
test_props = get_component_properties(test_comp, test_batch, props_dict, typical_dict)
print(f"Свойства для {test_comp}/{test_batch}: {len(test_props)}")
print(list(test_props.items())[:5])

## 5. Определение всех уникальных свойств

In [ ]:
# Собираем все уникальные названия свойств
all_props = set()
for d in list(props_dict.values()) + list(typical_dict.values()):
    all_props.update(d.keys())

all_props = sorted([p for p in all_props if pd.notna(p)])
print(f"Всего уникальных свойств: {len(all_props)}")

# Выбираем числовые свойства (те, которые можно преобразовать в числа)
numeric_props = []
for p in all_props:
    # Пробуем преобразовать в число
    values = []
    for d in list(props_dict.values()) + list(typical_dict.values()):
        if p in d:
            try:
                values.append(float(d[p].replace(',', '.')))
            except:
                pass
    if len(values) > 10:  #需要有 достаточное количество значений
        numeric_props.append(p)

print(f"Числовых свойств: {len(numeric_props)}")
print(numeric_props[:15])

## 6. Подготовка признаков сценария

In [ ]:
def safe_float(x):
    """Безопасное преобразование в float"""
    if pd.isna(x):
        return np.nan
    try:
        return float(str(x).replace(',', '.'))
    except:
        return np.nan

def prepare_scenario_features(scenario_df, scenario_id, props_dict, typical_dict, numeric_props):
    """Подготавливает признаки для одного сценария"""
    scenario_data = scenario_df[scenario_df[SCENARIO_COL] == scenario_id]
    
    # Параметры теста (одинаковые для всех компонентов сценария)
    test_params = scenario_data.iloc[0]
    
    features = {
        'temp': safe_float(test_params[TEMP_COL]),
        'time': safe_float(test_params[TIME_COL]),
        'bio': safe_float(test_params[BIO_COL]),
        'catalyst': safe_float(test_params[CAT_COL]),
    }
    
    # Собираем данные о компонентах
    components = []
    component_types = []
    masses = []
    
    for _, row in scenario_data.iterrows():
        comp = row[COMP_COL]
        batch = row[BATCH_COL] if pd.notna(row[BATCH_COL]) else ''
        mass = safe_float(row[MASS_COL])
        
        # Получаем свойства компонента
        comp_props = get_component_properties(comp, batch, props_dict, typical_dict)
        
        # Преобразуем в числовые признаки
        prop_features = {}
        for prop_name in numeric_props:
            if prop_name in comp_props:
                prop_features[prop_name] = safe_float(comp_props[prop_name])
            else:
                prop_features[prop_name] = np.nan
        
        components.append({
            'type': comp,
            'mass': mass,
            'properties': prop_features
        })
        component_types.append(comp)
        masses.append(mass)
    
    features['components'] = components
    features['component_types'] = component_types
    features['masses'] = masses
    features['n_components'] = len(components)
    
    # Агрегаты по массам
    masses_arr = np.array(masses)
    features['mass_mean'] = np.nanmean(masses_arr)
    features['mass_std'] = np.nanstd(masses_arr)
    features['mass_max'] = np.nanmax(masses_arr)
    features['mass_min'] = np.nanmin(masses_arr)
    
    return features

# Тест на одном сценарии
test_scenario = 'train_1'
test_features = prepare_scenario_features(train, test_scenario, props_dict, typical_dict, numeric_props)
print(f"Сценарий {test_scenario}: {test_features['n_components']} компонентов")
print(f"Параметры теста: temp={test_features['temp']}, time={test_features['time']}, bio={test_features['bio']}, catalyst={test_features['catalyst']}")

## 7. Подготовка данных для всех сценариев

In [ ]:
def encode_component_type(comp_type, le):
    """Кодирует тип компонента"""
    if comp_type in le:
        return le.transform([comp_type])[0]
    else:
        return -1

# Собираем все типы компонентов
all_component_types = set(train[COMP_COL].unique()) | set(test[COMP_COL].unique())
le = LabelEncoder()
le.fit(list(all_component_types))
print(f"Всего типов компонентов: {len(le.classes_)}")

In [ ]:
# Подготовка данных для всех сценариев TRAIN
train_scenarios = train[SCENARIO_COL].unique()
print(f"Подготовка {len(train_scenarios)} сценариев TRAIN...")

train_data = []
for i, sid in enumerate(train_scenarios):
    features = prepare_scenario_features(train, sid, props_dict, typical_dict, numeric_props)
    
    # Целевые переменные
    scenario_rows = train[train[SCENARIO_COL] == sid]
    target_visc = scenario_rows[TARGET_VISC].iloc[0]
    target_oxid = scenario_rows[TARGET_OXID].iloc[0]
    
    train_data.append({
        'scenario_id': sid,
        'features': features,
        'target_visc': target_visc,
        'target_oxid': target_oxid
    })
    
    if (i + 1) % 50 == 0:
        print(f"  Обработано {i+1}/{len(train_scenarios)}")

print(f"Готово: {len(train_data)} сценариев")

In [ ]:
# Подготовка данных для всех сценариев TEST
test_scenarios = test[SCENARIO_COL].unique()
print(f"Подготовка {len(test_scenarios)} сценариев TEST...")

test_data = []
for i, sid in enumerate(test_scenarios):
    features = prepare_scenario_features(test, sid, props_dict, typical_dict, numeric_props)
    
    test_data.append({
        'scenario_id': sid,
        'features': features
    })

print(f"Готово: {len(test_data)} сценариев")

## 8. Создание финальных массивов

In [ ]:
# Фиксируем порядок свойств для векторов
PROPS_ORDER = numeric_props[:20]  # Берем топ-20 числовых свойств
print(f"Используем свойства: {PROPS_ORDER}")

In [ ]:
def create_scenario_vector(data_item, le, props_order):
    """Создает вектор признаков для сценария"""
    f = data_item['features']
    
    # Глобальные признаки
    global_features = [
        f['temp'], f['time'], f['bio'], f['catalyst'],
        f['n_components'],
        f['mass_mean'], f['mass_std'], f['mass_max'], f['mass_min']
    ]
    
    # Признаки компонентов (unordered set - суммируем)
    comp_vectors = []
    for comp in f['components']:
        # Тип компонента
        type_enc = encode_component_type(comp['type'], le)
        # Масса
        mass = comp['mass'] if not np.isnan(comp['mass']) else 0
        # Свойства
        prop_vals = []
        for p in props_order:
            pv = comp['properties'].get(p, np.nan)
            prop_vals.append(pv if not np.isnan(pv) else 0)
        
        comp_vectors.append([type_enc, mass] + prop_vals)
    
    # Пуллим (сумма,均价, макс) - инвариант к порядку
    comp_arr = np.array(comp_vectors)
    if len(comp_arr) > 0:
        pooled = [
            np.nanmean(comp_arr, axis=0),
            np.nanmax(comp_arr, axis=0),
            np.nanmin(comp_arr, axis=0)
        ].flatten()
    else:
        pooled = np.zeros((3, len(props_order) + 2))
    
    return np.concatenate([global_features, pooled])

In [ ]:
# Создаем массивы для TRAIN
X_train = []
y_visc = []
y_oxid = []
train_ids = []

for item in train_data:
    vec = create_scenario_vector(item, le, PROPS_ORDER)
    X_train.append(vec)
    y_visc.append(item['target_visc'])
    y_oxid.append(item['target_oxid'])
    train_ids.append(item['scenario_id'])

X_train = np.array(X_train)
y_visc = np.array(y_visc)
y_oxid = np.array(y_oxid)

print(f"X_train: {X_train.shape}")
print(f"y_visc: {y_visc.shape}, y_oxid: {y_oxid.shape}")

In [ ]:
# Создаем массивы для TEST
X_test = []
test_ids = []

for item in test_data:
    vec = create_scenario_vector(item, le, PROPS_ORDER)
    X_test.append(vec)
    test_ids.append(item['scenario_id'])

X_test = np.array(X_test)
print(f"X_test: {X_test.shape}")

## 9. Нормализация

In [ ]:
# Заменяем NaN на 0 для нормализации
X_train_nan = np.nan_to_num(X_train, nan=0.0)
X_test_nan = np.nan_to_num(X_test, nan=0.0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_nan)
X_test_scaled = scaler.transform(X_test_nan)

print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"X_test_scaled: {X_test_scaled.shape}")

## 10. Сохранение

In [ ]:
# Сохраняем подготовленные данные
np.save(f'{DATA_PATH}/X_train.npy', X_train_scaled)
np.save(f'{DATA_PATH}/X_test.npy', X_test_scaled)
np.save(f'{DATA_PATH}/y_visc.npy', y_visc)
np.save(f'{DATA_PATH}/y_oxid.npy', y_oxid)

# Сохраняем ID
pd.DataFrame({'scenario_id': train_ids}).to_csv(f'{DATA_PATH}/train_ids.csv', index=False)
pd.DataFrame({'scenario_id': test_ids}).to_csv(f'{DATA_PATH}/test_ids.csv', index=False)

# Сохраняем label encoder и scaler
import pickle
with open(f'{DATA_PATH}/le.pkl', 'wb') as f:
    pickle.dump(le, f)
with open(f'{DATA_PATH}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(f'{DATA_PATH}/props_order.pkl', 'wb') as f:
    pickle.dump(PROPS_ORDER, f)

print("Данные сохранены!")